In [ ]:
import os
import mysql.connector
from dotenv import load_dotenv

# Cargamos las variables del archivo .env
load_dotenv()

if not os.getenv("DB_PORT"):
    print("⚠️ ATENCIÓN: No se pudo leer el archivo .env.")
else:
    try:
        # Intentar conectar a la base de datos
        conexion = mysql.connector.connect(
            host=os.getenv("DB_HOST"),
            port=os.getenv("DB_PORT"),
            database=os.getenv("DB_NAME"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD")
        )
        
        if conexion.is_connected():
            print("¡Conexión exitosa a la base de datos BiblioIA!")
            
            # Acá está el cambio: usamos server_info sin paréntesis
            info_servidor = conexion.server_info 
            print(f"Versión del servidor MySQL: {info_servidor}")
            
            conexion.close()
            
    except mysql.connector.Error as err:
        print(f"Error al conectar a MySQL: {err}")

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Cargamos el .env
load_dotenv(override=True)

# Configuramos el cliente para Groq
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("LLM_API_KEY")
)

# Prueba de conexión actualizada
print("Probando conexión a Groq...")
try:
    chat_completion = client.chat.completions.create(
        messages=[{"role": "user", "content": "Hola, ¿estás ahí?"}],
        # Usamos un modelo que sí está activo actualmente
        model="llama-3.3-70b-versatile",
    )
    print("🤖 Respuesta de la IA:")
    print(chat_completion.choices[0].message.content)
except Exception as e:
    print(f"❌ Error: {e}")

In [5]:
import os
import pandas as pd
import mysql.connector
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display

# 1. Cargamos las credenciales
load_dotenv(override=True)

# 2. Configuramos la IA (Groq)
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("LLM_API_KEY")
)

# 3. Función para que la IA traduzca de Español a SQL
def consultar_biblioia(pregunta_usuario):
    prompt_sistema = """
    Sos un asistente experto en bases de datos MySQL para una biblioteca. 
    Tu objetivo es traducir preguntas en lenguaje natural a consultas SQL precisas.

    ESQUEMA DE LA BASE DE DATOS:
    - Libro (Isbn, Titulo, Año, StockTotal, StockDisponible)
    - Socio (Id, Dni, Nombre, Apellido, Mail, FechaAlta, IdSexo, IdEstadoSocio)
    - Ejemplar (Numero,FechaAlta, IsbnLibro, IdEstadoEjemplar)
    - Prestamo (Id,FechaPrestamo, FechaVencimiento, FechaDevolucion, IdSocio, IdEjemplar, IdEstadoPrestamo)
    - Sancion (Id, FechaInicio, FechaFin, Motivo, IdTipoSancion, IdPrestamo, IdSocio)
    - GeneroLibro(Id, Nombre, Descripcion)
    - Autor(Id, Nombre, Apellido,IdSexo, IdNacionalidad)
    - Nacionalidad(Id, Pais)
    - Autor_Libro(IdAutor, IsbnLibro)
    - GeneroLibro_Libro(IsbnLibro, IdGeneroLibro)
    
    CONSTRAINTS Y ESTADOS:
    - EstadoSocio: 1='Activo', 2='Suspendido', 3='Baja'
    - EstadoEjemplar: 1='Disponible', 2='Prestado', 3='Baja'
    - EstadoPrestamo: 1='Activo', 2='Devuelto', 3='Vencido'
    - TipoSancion: 1='Leve', 2='Medio', 3='Grave'
    - Sexo:  1='Femenino',2='Masculino', 3='Otro'

    EJEMPLOS:
    Pregunta: "¿Cuáles son los 5 libros más prestados?"
    Respuesta: SELECT l.Titulo, COUNT(p.Id) AS TotalPrestamos FROM Libro l JOIN Ejemplar e ON l.Isbn = e.IsbnLibro JOIN Prestamo p ON e.Numero = p.IdEjemplar GROUP BY l.Isbn, l.Titulo ORDER BY TotalPrestamos DESC LIMIT 5;

    INSTRUCCIÓN CRÍTICA:
    Respondé ÚNICA Y EXCLUSIVAMENTE con una consulta SQL válida para MySQL. Devuelve solo la cadena de texto de la consulta, sin bloques de código Markdown ni explicaciones.
    """
    
    try:
        chat_completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": prompt_sistema},
                {"role": "user", "content": pregunta_usuario}
            ],
            temperature=0.1
        )
        sql_generado = chat_completion.choices[0].message.content.strip()
        return sql_generado.replace("```sql", "").replace("```", "").strip()
    except Exception as e:
        return f"Error al generar SQL: {e}"

# 4. Función para ir a buscar los datos a MySQL
def ejecutar_consulta(sql):
    try:
        conexion = mysql.connector.connect(
            host=os.getenv("DB_HOST"),
            port=os.getenv("DB_PORT"),
            database=os.getenv("DB_NAME"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD")
        )
        df = pd.read_sql(sql, conexion)
        conexion.close()
        return df
    except Exception as e:
        return f"Error de MySQL: {e}"

# 5. El Agente Principal que une todo
def agente_responder(pregunta):
    print(f"👤 Pregunta: {pregunta}")
    print("🤖 Pensando y generando SQL...")
    
    sql_generado = consultar_biblioia(pregunta)
    print(f"⚙️ SQL a ejecutar:\n{sql_generado}\n")
    
    if sql_generado.startswith("Error"):
        print("❌ Hubo un problema al generar el código SQL.")
        return
        
    print("📊 Buscando datos en la base de datos...")
    resultado = ejecutar_consulta(sql_generado)
    
    print("\n--- RESULTADOS ---")
    if isinstance(resultado, pd.DataFrame):
        if resultado.empty:
            print("⚠️ La consulta fue exitosa, pero no encontró datos que coincidan.")
        else:
            display(resultado)
    else:
        print(f"❌ {resultado}")

# --- ¡PRUEBA FINAL! ---
def iniciar_demo():
    print("="*50)
    print("📚🤖 BIENVENIDO A BIBLIO-IA 🤖📚")
    print("Haceme cualquier pregunta sobre la biblioteca.")
    print("Escribí 'salir' para terminar.")
    print("="*50)
    
    while True:
        pregunta = input("\n👤 Tu pregunta: ")
        
        if pregunta.lower() in ['salir', 'exit', 'quit']:
            print("🤖 ¡Hasta luego! Gracias por usar BiblioIA.")
            break
            
        # Llamamos a nuestro agente mágico
        agente_responder(pregunta)

# Ejecutamos el chat
iniciar_demo()

📚🤖 BIENVENIDO A BIBLIO-IA 🤖📚
Haceme cualquier pregunta sobre la biblioteca.
Escribí 'salir' para terminar.
👤 Pregunta: ¿Qué libros de ciencia ficción están disponibles para prestar?
🤖 Pensando y generando SQL...
⚙️ SQL a ejecutar:
SELECT DISTINCT l.Titulo, l.Isbn FROM Libro l JOIN GeneroLibro_Libro gl ON l.Isbn = gl.IsbnLibro JOIN GeneroLibro g ON gl.IdGeneroLibro = g.Id WHERE g.Nombre = 'Ciencia Ficcion' AND l.StockDisponible > 0;

📊 Buscando datos en la base de datos...


C:\Users\santi\AppData\Local\Temp\ipykernel_4560\346231748.py:74: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)



--- RESULTADOS ---


,Titulo,Isbn
0,1984,9.780452e+12
1,El camino de los reyes,9.780765e+12
2,Palabras radiantes,9.780765e+12
3,Juramentada,9.781250e+12
4,El ritmo de la guerra,9.781250e+12
5,Fahrenheit 451,9.788434e+12
6,Fundación,9.788448e+12
7,"Yo, robot",9.788448e+12


👤 Pregunta: ¿Cuál es el historial de préstamos del socio con DNI 30123456?
🤖 Pensando y generando SQL...
⚙️ SQL a ejecutar:
SELECT p.FechaPrestamo, p.FechaVencimiento, p.FechaDevolucion, l.Titulo, e.Numero 
FROM Prestamo p 
JOIN Socio s ON p.IdSocio = s.Id 
JOIN Ejemplar e ON p.IdEjemplar = e.Numero 
JOIN Libro l ON e.IsbnLibro = l.Isbn 
WHERE s.Dni = 30123456 
ORDER BY p.FechaPrestamo DESC;

📊 Buscando datos en la base de datos...


...

👤 Pregunta: Quien es el socio que mas libros saco en prestamos el ultimo mes
🤖 Pensando y generando SQL...
⚙️ SQL a ejecutar:
SELECT s.Nombre, s.Apellido, COUNT(p.Id) AS TotalPrestamos 
FROM Socio s 
JOIN Prestamo p ON s.Id = p.IdSocio 
WHERE p.FechaPrestamo >= NOW() - INTERVAL 1 MONTH 
GROUP BY s.Id, s.Nombre, s.Apellido 
ORDER BY TotalPrestamos DESC 
LIMIT 1;

📊 Buscando datos en la base de datos...


C:\Users\santi\AppData\Local\Temp\ipykernel_4560\346231748.py:74: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)



--- RESULTADOS ---


,Nombre,Apellido,TotalPrestamos
0,Diego,Maradona,5


🤖 ¡Hasta luego! Gracias por usar BiblioIA.
